# Workshop Agent 2 — The Knowledge Retrieval / RAG Agent

**Book:** *30 Agents Every AI Engineer Must Build* by Imran Ahmad (Packt, 2026)
**Reference:** Chapter 6, §6.1 — Knowledge Retrieval Agents (pp. 146–153)

In this hands-on session you will build an end-to-end Retrieval-Augmented Generation (RAG) pipeline: load and chunk documents, embed them, store them in a FAISS vector index, and answer questions grounded only in that corpus — with source passages cited for provenance. You will see the four-stage modular architecture of Figure 6.1 (Query Understanding, Retrieval, Preprocessing, Synthesis) take shape in code, with the parallel Provenance component tracking citations throughout. A diagnostic query on a refund-policy scenario then demonstrates how source inspection exposes retrieval failures. A deep dive on chunking strategies (fixed-size, recursive, semantic) closes the loop on the most consequential configuration decision in a RAG system.

The notebook runs in **Simulation Mode** by default — no API key required. All outputs use chapter-derived mocks from `agent_utils.py` and are pedagogically equivalent to Live Mode.

In [ ]:
# Google Colab bootstrap — runs only on Colab, no-op everywhere else.
# Locally you are already inside the agent folder with requirements installed.
import os
import sys

if "google.colab" in sys.modules:
    AGENT_DIR = "02-knowledge-retrieval-rag-agent"
    if not os.path.exists("/content/repo"):
        os.system("git clone --depth 1 https://github.com/cloudanum/ws-10-agents /content/repo")
    os.chdir(f"/content/repo/{AGENT_DIR}")
    # On Colab, prefer requirements-colab.txt when present: it drops pins that
    # cannot coexist with Colab's preinstalled stack (e.g. langchain 0.2.16
    # requires numpy<2 on Python 3.13, while Colab ships numpy 2.x).
    req_file = "requirements-colab.txt" if os.path.exists("requirements-colab.txt") else "requirements.txt"
    # Keep Colab's preinstalled scientific/kernel stack: the kernel already has
    # numpy, pandas, pydantic and ipykernel loaded, so letting pip replace them
    # (e.g. building numpy 1.26.4 from source or upgrading ipykernel) breaks the
    # running kernel with ABI errors or an OOM kill. Filter those lines out of
    # requirements and constrain the rest of the install to the installed versions.
    import re
    from importlib.metadata import PackageNotFoundError, version
    filtered = [
        line for line in open(req_file)
        if not re.match(r"\s*(numpy|pandas|pydantic|jupyter|ipykernel)\b", line, re.IGNORECASE)
    ]
    with open("/tmp/colab_requirements.txt", "w") as fh:
        fh.writelines(filtered)
    pins = []
    for pkg in ("numpy", "pandas", "pydantic", "ipykernel"):
        try:
            pins.append(f"{pkg}=={version(pkg)}")
        except PackageNotFoundError:
            pass
    with open("/tmp/colab_constraints.txt", "w") as fh:
        fh.write("\n".join(pins) + "\n")
    import subprocess
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", "/tmp/colab_requirements.txt",
         "--constraint", "/tmp/colab_constraints.txt"],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print("pip install failed — re-running without -q for the full resolver report:\n")
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "-r", "/tmp/colab_requirements.txt",
             "--constraint", "/tmp/colab_constraints.txt"],
        )
        raise RuntimeError("Colab bootstrap: pip install failed (see resolver report above)")
    print(f"Colab setup complete ({req_file}) — working directory: {os.getcwd()}")
else:
    print("Not on Colab — skipping bootstrap (local setup already in place).")

## 0. Setup & Configuration

This section initializes the environment, detects API keys, and sets the execution mode.

**Execution Modes:**
- **Simulation Mode** (default): No API key required. All outputs use chapter-derived mocks from `agent_utils.py`. Pedagogically equivalent to live output.
- **Live Mode**: Requires a valid `OPENAI_API_KEY` in `.env`. Makes real API calls to OpenAI, live arXiv queries, and real OCR processing.

> **Ref:** See `AGENTS.md` for the full capability declaration and persona prompt.

In [1]:
# ── 0.1 Dependency Check ─────────────────────────────────────────
# Ref: requirements.txt
# Verify core packages are available before proceeding.

import importlib
import sys

REQUIRED = [
    "dotenv", "numpy", "pandas", "langchain", "langchain_openai",
    "langchain_community", "langchain_text_splitters", "faiss",
    "PIL", "rapidfuzz", "sklearn",
]

missing = []
for pkg in REQUIRED:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"⚠️  Missing packages: {missing}")
    print("   Run: pip install -r requirements.txt")
else:
    print("✅ All core dependencies available.")

print(f"   Python {sys.version}")

⚠️  Missing packages: ['langchain_openai', 'PIL', 'rapidfuzz', 'sklearn']
   Run: pip install -r requirements.txt
   Python 3.12.6 (v3.12.6:a4a2d2b0d85, Sep  6 2024, 16:08:03) [Clang 13.0.0 (clang-1300.0.29.30)]


In [2]:
# Multi-provider LLM support (OpenAI / Anthropic / Google Gemini)
# Set LLM_PROVIDER in .env to choose: openai | anthropic | google | auto
# Auto-detection uses the first available key.
# See supporting/llm_provider.py for details.

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')

try:
    from supporting.llm_provider import detect_provider, get_llm, PROVIDER_MODELS, print_provider_banner
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = detect_provider()
    print_provider_banner(_PROVIDER, _PROVIDER_MODE)
except ImportError:
    print('[INFO] supporting/llm_provider.py not found — using default OpenAI path')
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = 'openai', os.getenv('OPENAI_API_KEY'), 'LIVE' if os.getenv('OPENAI_API_KEY') else 'SIMULATION'



   SIMULATION MODE ACTIVE
   Using MockLLM — no API key required



In [3]:
# ── 0.2 Import Shared Utilities ──────────────────────────────────
# Ref: agent_utils.py — ColorLogger, fail_gracefully, MockLLM, etc.

import os
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from agent_utils import (
    ColorLogger, log, fail_gracefully, get_api_key,
    MockLLM, MockRetrievalQAResult, MockEmbeddings,
    MockOcrToken, MOCK_INVOICE_TOKENS, MOCK_EXTRACTED_FIELDS,
    mock_pytesseract_output, MOCK_ARXIV_PAPERS, mock_search_arxiv,
)

log.success("agent_utils loaded — all shared utilities available.")

[SUCCESS] agent_utils loaded — all shared utilities available.


In [4]:
# ── 0.3 API Key Detection & Mode Selection ───────────────────────
# Ref: Zero-Hardcode Policy
# Cascade: .env → os.getenv → getpass → SIMULATION MODE

api_key = get_api_key("OPENAI_API_KEY")
SIMULATION_MODE = api_key is None

# ── Mode Banner ──────────────────────────────────────────────────
if SIMULATION_MODE:
    print()
    print("=" * 65)
    print("  🔬  SIMULATION MODE ACTIVE")
    print("  All outputs are chapter-derived mocks from agent_utils.py.")
    print("  To enable Live Mode, add OPENAI_API_KEY to .env")
    print("=" * 65)
else:
    print()
    print("=" * 65)
    try:
        sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '..'))
        from supporting.llm_provider import detect_provider, print_provider_banner
        _prov = os.environ.get("LLM_PROVIDER", "auto")
        if _prov == "auto":
            _prov, _, _ = detect_provider()
        print_provider_banner(_prov, "LIVE")
    except ImportError:
        print("  🌐  LIVE MODE ACTIVE")
        print("  Real API calls enabled.")
    print("=" * 65)

[INFO]    No valid OPENAI_API_KEY found in .env or environment.
[ERROR]   No API key available for OPENAI_API_KEY. SIMULATION MODE activated — all outputs are chapter-derived mocks.

  🔬  SIMULATION MODE ACTIVE
  All outputs are chapter-derived mocks from agent_utils.py.
  To enable Live Mode, add OPENAI_API_KEY to .env


---

## 1. Knowledge Retrieval Agent (pp. 146–153)

**Ref:** §6.1 — Knowledge Retrieval Agents (pp. 146–153)

A Knowledge Retrieval agent is the lifeline connecting an LLM's static training data to the living, ever-changing world of information. By linking to live sources such as databases and APIs, these agents directly address two critical LLM weaknesses: **knowledge cutoff** and **hallucination risk**, anchoring outputs in verifiable evidence.

### Modular Architecture (Figure 6.1, p. 148)

The agent operates through four sequential stages with parallel provenance tracking:

1. **Query Understanding** — Parse intent, disambiguate, reformulate into a precise search query
2. **Retrieval** — Execute the search plan against vector databases or search APIs (lexical, semantic, or hybrid)
3. **Preprocessing** — Chunk documents, generate embeddings, filter irrelevant results
4. **Synthesis** — Integrate retrieved content into the LLM prompt; generate a grounded answer with provenance

> **Architecture:** See Figure 6.1 (p. 148) for the full modular architecture diagram showing the parallel **Provenance** component that collects citations, metadata, and confidence metrics throughout the pipeline.

### Implementation Patterns (pp. 148–149)

Three retrieval workflow patterns exist, each suited to different query complexity:
- **Single-stage retrieval** — Direct query to one source. Low latency, limited recall.
- **Multi-stage retrieval** — Broad search refined through targeted filters. Higher latency, better for exploratory queries.
- **Hybrid retrieval** — Combines keyword (lexical/BM25) and vector similarity (semantic) search. Best recall for mixed-content corpora.

> **Note — Agent Capability Level** (p. 146): A Knowledge Retrieval agent typically operates at **Level 2** (Tool-Using agent), parsing requests and chaining tool operations. More advanced agents that decompose high-level goals and maintain memory across steps can exhibit **Level 3** (Planning agent) behaviors.

In this section, we implement an end-to-end RAG pipeline using LangChain, OpenAI embeddings (or mock equivalents), and FAISS as the vector store.


### Figure 6.1 — Modular Architecture of a Knowledge Retrieval Agent (p. 148)

```
                    ┌────────────────────────────────────┐
  User Query ──────▶│  Query Understanding Layer         │
                    │  Intent parsing, disambiguation,   │
                    │  query reformulation               │
                    └────────────────┬───────────────────┘
                                     ▼
 ┌─────────────┐   ┌────────────────────────────────────┐   ┌──────────────────┐
 │ Vector DB   │──▶│  Retriever Module                  │   │  Provenance       │
 │ Search API  │──▶│  Lexical, semantic, or hybrid      │   │  ┌──────────────┐│
 │ Relational  │──▶│  retrieval from multiple sources   │   │  │ Citations    ││
 └─────────────┘   └────────────────┬───────────────────┘   │  │ Metadata     ││
                                     ▼                       │  │ Traceability ││
                    ┌────────────────────────────────────┐   │  │ Confidence   ││
                    │  Preprocessing                     │◀──│  │ metrics      ││
                    │  Chunking, embedding generation,   │   │  └──────────────┘│
                    │  filtering, deduplication          │   └──────────────────┘
                    └────────────────┬───────────────────┘
                                     ▼
                    ┌────────────────────────────────────┐
                    │  Reasoning and Generation          │
                    │  Synthesis within LLM context      │──────▶ Grounded Answer
                    │  using only retrieved sources      │
                    └────────────────────────────────────┘
```


In [5]:
# ── 1.1 Load Documents ────────────────────────────────────────────
# Ref: §6.1, RAG Pipeline Step 2 — DirectoryLoader (p. 149)
#
# We load from the docs/ directory which contains:
#   - knowledge_base_rag.txt: RAG concepts, strategies, limitations
#   - compliance_policy.txt:  Corporate policy (data retention, refunds)

from langchain_text_splitters import RecursiveCharacterTextSplitter

log.info("Loading documents from docs/ directory...")

doc_dir = "docs"
documents = []

for fname in sorted(os.listdir(doc_dir)):
    fpath = os.path.join(doc_dir, fname)
    if os.path.isfile(fpath) and fname.endswith(".txt"):
        with open(fpath, "r", encoding="utf-8") as f:
            content = f.read()
        documents.append({"content": content, "source": fpath})
        log.info(f"  Loaded: {fname} ({len(content):,} chars)")

log.success(f"Loaded {len(documents)} documents from {doc_dir}/")

[INFO]    Loading documents from docs/ directory...
[INFO]      Loaded: compliance_policy.txt (2,816 chars)
[INFO]      Loaded: knowledge_base_rag.txt (7,237 chars)
[SUCCESS] Loaded 2 documents from docs/


In [6]:
# ── 1.2 Split Documents into Chunks ───────────────────────────────
# Ref: §6.1, Chunking Strategies (p. 151)
#
# Parameters from the chapter's RAG pipeline example (p. 149):
#   chunk_size=1000, chunk_overlap=200
#
# RecursiveCharacterTextSplitter splits on natural boundaries
# (paragraphs → sentences → words) in descending order.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

all_chunks = []
all_metadatas = []

for doc in documents:
    chunks = splitter.split_text(doc["content"])
    for chunk in chunks:
        all_chunks.append(chunk)
        all_metadatas.append({"source": doc["source"]})

log.success(
    f"Split {len(documents)} documents into {len(all_chunks)} chunks "
    f"(chunk_size=1000, overlap=200)"
)

# Preview first chunk
print(f"\n--- Chunk 0 preview (first 200 chars) ---")
print(all_chunks[0][:200] + "...")
print(f"Source: {all_metadatas[0]['source']}")

[SUCCESS] Split 2 documents into 14 chunks (chunk_size=1000, overlap=200)

--- Chunk 0 preview (first 200 chars) ---
Acme Corporation — Corporate Compliance and Data Governance Policy
Effective Date: January 1, 2026
Document Classification: Internal — All Employees

1. Data Retention

All financial records, includin...
Source: docs/compliance_policy.txt


In [7]:
# ── 1.3 Create Embeddings & FAISS Vector Store ────────────────────
# Ref: §6.1, Step 3 — OpenAIEmbeddings + FAISS.from_texts (pp. 149–150)
#
# In Simulation Mode: MockEmbeddings produces deterministic 256-dim
# vectors via seeded hashing — no API key needed.
# In Live Mode: OpenAIEmbeddings with text-embedding-3-large.

import numpy as np

if SIMULATION_MODE:
    log.info("[SIMULATION MODE] Using MockEmbeddings (256-dim, hash-seeded)")
    embeddings = MockEmbeddings()
else:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
    log.info("[LIVE MODE] Using OpenAIEmbeddings (text-embedding-3-large)")

# Build FAISS index
from langchain_community.vectorstores import FAISS

@fail_gracefully(fallback_return=None, section_ref="6.1")
def build_faiss_index(chunks, metadatas, embed_model):
    """Create FAISS vector store from document chunks."""
    vectorstore = FAISS.from_texts(
        texts=chunks,
        embedding=embed_model,
        metadatas=metadatas,
    )
    return vectorstore

vectorstore = build_faiss_index(all_chunks, all_metadatas, embeddings)

if vectorstore is not None:
    log.success(f"FAISS index built with {len(all_chunks)} vectors.")
else:
    log.error("FAISS index creation failed — will use MockRetrievalQAResult.")

[INFO]    [SIMULATION MODE] Using MockEmbeddings (256-dim, hash-seeded)


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


[INFO]    Executing: build_faiss_index [Ref: §6.1]
[SUCCESS] build_faiss_index completed. [Ref: §6.1]
[SUCCESS] FAISS index built with 14 vectors.


In [8]:
# ── 1.4 Build Retrieval + Generation Chain ────────────────────────
# Ref: §6.1, Step 4 — RetrievalQA.from_chain_type (p. 150)
#
# The retriever returns the top k=3 most similar chunks.
# These chunks are passed to the LLM as context for grounded generation.

@fail_gracefully(fallback_return=None, section_ref="6.1")
def build_qa_chain(vstore, simulation_mode):
    """Build RetrievalQA chain with real or mock LLM."""
    if simulation_mode or vstore is None:
        return None  # Will use MockRetrievalQAResult instead

    from langchain_openai import ChatOpenAI
    from langchain.chains import RetrievalQA

    retriever = vstore.as_retriever(search_kwargs={"k": 3})
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        return_source_documents=True,
    )
    return qa_chain

qa_chain = build_qa_chain(vectorstore, SIMULATION_MODE)

if qa_chain is not None:
    log.success("RetrievalQA chain ready (Live Mode).")
else:
    log.info("[SIMULATION MODE] Using MockRetrievalQAResult for queries.")

[INFO]    Executing: build_qa_chain [Ref: §6.1]
[SUCCESS] build_qa_chain completed. [Ref: §6.1]
[INFO]    [SIMULATION MODE] Using MockRetrievalQAResult for queries.


In [9]:
# ── 1.5 Run a Query — Grounded Answer with Provenance ─────────────
# Ref: §6.1, Step 5 — Query execution (p. 150)
#
# Query: "What are the main limitations of retrieval-augmented generation?"
# This matches the exact query from the chapter's RAG pipeline example.

query = "What are the main limitations of retrieval-augmented generation?"
log.info(f"Query: {query}")
print()

if qa_chain is not None:
    # Live Mode: use real RetrievalQA chain
    result = qa_chain({"query": query})
    answer = result["result"]
    sources = result.get("source_documents", [])
else:
    # Simulation Mode: use chapter-derived mock
    mock_result = MockRetrievalQAResult(query).run()
    answer = mock_result["result"]
    sources = mock_result["source_documents"]

# ── Display Answer ────────────────────────────────────────────────
print("=" * 65)
print("ANSWER:")
print("=" * 65)
print(answer)

# ── Display Sources (Provenance) ──────────────────────────────────
print()
print("SOURCES:")
print("-" * 40)
for i, doc in enumerate(sources, 1):
    if isinstance(doc, dict):
        src = doc.get("metadata", {}).get("source", "unknown")
    else:
        src = getattr(doc, "metadata", {}).get("source", "unknown")
    print(f"  [{i}] {src}")

log.success("Knowledge Retrieval Agent query completed with provenance.")

[INFO]    Query: What are the main limitations of retrieval-augmented generation?

ANSWER:
[SIMULATION MODE] Based on the retrieved documents, the main limitations of retrieval-augmented generation include: (1) Noise in retrieved chunks can degrade answer quality — irrelevant context dilutes the LLM's focus (§6.1, Noise reduction). (2) Index freshness — if the vector store is not regularly updated, answers reflect stale information (§6.1, Index freshness). (3) Latency overhead — the retrieval step adds response time compared to direct generation (§6.1, Latency control). (4) Chunking sensitivity — poor chunk_size or overlap parameters can split key facts across boundaries, causing incomplete answers (§6.1, Chunking strategies).

Sources: docs/knowledge_base_rag.txt (simulated)

SOURCES:
----------------------------------------
  [1] docs/knowledge_base_rag.txt
  [2] docs/compliance_policy.txt
  [3] docs/knowledge_base_rag.txt
[SUCCESS] Knowledge Retrieval Agent query completed with prov

In [10]:
# ── 1.6 Diagnostic Query — Refund Policy Scenario ─────────────────
# Ref: §6.1, Diagnosing Retrieval Failures (p. 152)
#
# The chapter describes a scenario: a user asks "What is our refund
# policy for subscriptions?" and gets generic billing terms instead of
# the specific subscription clause. This demonstrates why source
# inspection and metadata filtering matter.

diag_query = "What is our refund policy for subscriptions?"
log.info(f"Diagnostic query: {diag_query}")
print()

if qa_chain is not None:
    result = qa_chain({"query": diag_query})
    answer = result["result"]
    sources = result.get("source_documents", [])
else:
    mock_result = MockRetrievalQAResult(diag_query).run()
    answer = mock_result["result"]
    sources = mock_result["source_documents"]

print("=" * 65)
print("ANSWER:")
print("=" * 65)
print(answer)
print()
print("SOURCES:")
print("-" * 40)
for i, doc in enumerate(sources, 1):
    if isinstance(doc, dict):
        src = doc.get("metadata", {}).get("source", "unknown")
    else:
        src = getattr(doc, "metadata", {}).get("source", "unknown")
    print(f"  [{i}] {src}")

log.success("Diagnostic query completed — inspect sources for retrieval quality.")

[INFO]    Diagnostic query: What is our refund policy for subscriptions?

ANSWER:
[SIMULATION MODE] The subscription refund policy allows full refunds within 14 days of renewal. After 14 days, refunds are prorated based on remaining subscription period.

Sources: docs/compliance_policy.txt (simulated)

SOURCES:
----------------------------------------
  [1] docs/knowledge_base_rag.txt
  [2] docs/compliance_policy.txt
  [3] docs/knowledge_base_rag.txt
[SUCCESS] Diagnostic query completed — inspect sources for retrieval quality.


---

## 2. Chunking Strategies Deep Dive (p. 151)

**Ref:** §6.1 — Chunking Strategies (p. 151)

Chunking is the **most consequential configuration decision** in a RAG system. The chunk is the atomic unit retrieved by the vector index; its size determines how much text the LLM receives as context for each match.

### Three Strategies (p. 151)

1. **Fixed-size chunking** — Splits at a fixed character/token boundary. Simplest approach; suits uniform documents.
2. **Recursive chunking** — Splits on natural boundaries (paragraphs → sentences → words) in descending order. **Recommended default** for mixed-content corpora.
3. **Semantic chunking** — Uses embedding similarity to detect topic shifts. Highest retrieval fidelity for narrative text; higher computational cost at ingestion time.

### The Size-Overlap Trade-Off (p. 151)

- **Smaller chunks** (200–500 chars): better precision, but risk losing surrounding context
- **Larger chunks** (1,000–2,000 chars): richer context, but diluted embedding signal reducing recall
- **Overlap** (e.g., 200 chars on 1,000-char chunks): ensures boundary sentences are captured by at least one chunk

> **Production Warning** (p. 151): Misconfiguring chunk size and overlap is the most common source of retrieval-quality degradation in production. Overly large chunks introduce irrelevant context, overly small chunks produce incomplete answers, and insufficient overlap creates boundary artifacts where key facts fall between chunks.


In [11]:
# ── 2.1 Sample Text for Chunking Comparison ───────────────────────
# Ref: §6.1, Chunking Strategies (p. 151)
#
# We use the first document from our corpus to demonstrate all three
# chunking strategies side by side.

with open("docs/knowledge_base_rag.txt", "r") as f:
    sample_text = f.read()

log.info(f"Sample text length: {len(sample_text):,} characters")
print(f"First 200 chars: {sample_text[:200]}...")

[INFO]    Sample text length: 7,237 characters
First 200 chars: Retrieval-Augmented Generation: Principles, Strategies, and Operational Considerations

Overview

Retrieval-Augmented Generation (RAG) is a methodology that merges information retrieval
with generativ...


In [12]:
# ── 2.2 Fixed-Size Chunking ────────────────────────────────────────
# Ref: §6.1, "Fixed-size chunking divides text at a fixed character
#       or token boundary" (p. 151)

from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="",           # Pure character boundary
    chunk_size=500,
    chunk_overlap=0,        # No overlap for contrast
)
fixed_chunks = fixed_splitter.split_text(sample_text)

log.info(f"Fixed-size chunking: {len(fixed_chunks)} chunks (size=500, overlap=0)")
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f'  Chunk {i}: {len(chunk)} chars | "{chunk[:60]}..."')

print(f"  ... ({len(fixed_chunks)} total chunks)")

[INFO]    Fixed-size chunking: 15 chunks (size=500, overlap=0)
  Chunk 0: 500 chars | "Retrieval-Augmented Generation: Principles, Strategies, and ..."
  Chunk 1: 500 chars | "ta, RAG transforms static language models into
dynamic, evid..."
  Chunk 2: 500 chars | "o manageable chunks, embedded as semantic vectors, and
filte..."
  ... (15 total chunks)


In [13]:
# ── 2.3 Recursive Chunking (Recommended Default) ──────────────────
# Ref: §6.1, "Recursive chunking attempts to split on natural
#       boundaries (paragraphs, sentences, words)" (p. 151)
#
# Parameters match the chapter's RAG pipeline: chunk_size=1000, overlap=200

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
recursive_chunks = recursive_splitter.split_text(sample_text)

log.info(f"Recursive chunking: {len(recursive_chunks)} chunks (size=1000, overlap=200)")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f'  Chunk {i}: {len(chunk)} chars | "{chunk[:60]}..."')

print(f"  ... ({len(recursive_chunks)} total chunks)")

# Demonstrate overlap preservation
if len(recursive_chunks) >= 2:
    tail = recursive_chunks[0][-50:]
    head = recursive_chunks[1][:50]
    overlap_found = any(
        tail[i:i+20] in recursive_chunks[1][:250]
        for i in range(len(tail) - 20)
    )
    print(f"\n  Overlap check (chunk 0 tail → chunk 1 head):")
    print(f"    Chunk 0 ends:   ...{repr(recursive_chunks[0][-60:])}")
    print(f"    Chunk 1 starts: {repr(recursive_chunks[1][:60])}...")
    print(f"    Overlap preserved: {'Yes ✓' if overlap_found else 'Minimal (natural boundary split)'}")

[INFO]    Recursive chunking: 10 chunks (size=1000, overlap=200)
  Chunk 0: 588 chars | "Retrieval-Augmented Generation: Principles, Strategies, and ..."
  Chunk 1: 872 chars | "A RAG system operates in a continuous cycle. First, a Query ..."
  Chunk 2: 810 chars | "Retrieval Strategies

The design of the retrieval workflow d..."
  ... (10 total chunks)

  Overlap check (chunk 0 tail → chunk 1 head):
    Chunk 0 ends:   ...'nguage models into\ndynamic, evidence-grounded collaborators.'
    Chunk 1 starts: 'A RAG system operates in a continuous cycle. First, a Query '...
    Overlap preserved: Minimal (natural boundary split)


In [14]:
# ── 2.4 Semantic Chunking (Simulated) ─────────────────────────────
# Ref: §6.1, "Semantic chunking uses embedding similarity to detect
#       natural topic shifts before splitting" (p. 151)
#
# In a production system, this would use an embedding model to compute
# similarity between adjacent sentences and split where similarity
# drops below a threshold. Here we simulate the concept.

log.info("Semantic chunking (simulated via paragraph boundaries)")

# Approximate semantic chunking by splitting on double-newlines (paragraphs)
# then merging adjacent paragraphs that are semantically related (by length heuristic)
paragraphs = [p.strip() for p in sample_text.split("\n\n") if p.strip()]

semantic_chunks = []
current_chunk = ""
for para in paragraphs:
    if len(current_chunk) + len(para) < 1200:
        current_chunk += ("\n\n" + para if current_chunk else para)
    else:
        if current_chunk:
            semantic_chunks.append(current_chunk)
        current_chunk = para
if current_chunk:
    semantic_chunks.append(current_chunk)

log.info(f"Semantic chunking: {len(semantic_chunks)} chunks (paragraph-aligned)")
for i, chunk in enumerate(semantic_chunks[:3]):
    print(f'  Chunk {i}: {len(chunk)} chars | "{chunk[:60]}..."')

print(f"  ... ({len(semantic_chunks)} total chunks)")

[INFO]    Semantic chunking (simulated via paragraph boundaries)
[INFO]    Semantic chunking: 8 chunks (paragraph-aligned)
  Chunk 0: 588 chars | "Retrieval-Augmented Generation: Principles, Strategies, and ..."
  Chunk 1: 1031 chars | "A RAG system operates in a continuous cycle. First, a Query ..."
  Chunk 2: 1032 chars | "Single-stage retrieval issues a direct query to a single aut..."
  ... (8 total chunks)


In [15]:
# ── 2.5 Chunking Comparison Summary ────────────────────────────────
# Ref: §6.1, Chunking Strategies (p. 151)

import pandas as pd

comparison = pd.DataFrame({
    "Strategy": ["Fixed-size", "Recursive (recommended)", "Semantic (simulated)"],
    "Chunks": [len(fixed_chunks), len(recursive_chunks), len(semantic_chunks)],
    "Avg Size (chars)": [
        int(sum(len(c) for c in fixed_chunks) / max(len(fixed_chunks), 1)),
        int(sum(len(c) for c in recursive_chunks) / max(len(recursive_chunks), 1)),
        int(sum(len(c) for c in semantic_chunks) / max(len(semantic_chunks), 1)),
    ],
    "Min Size": [
        min(len(c) for c in fixed_chunks),
        min(len(c) for c in recursive_chunks),
        min(len(c) for c in semantic_chunks),
    ],
    "Max Size": [
        max(len(c) for c in fixed_chunks),
        max(len(c) for c in recursive_chunks),
        max(len(c) for c in semantic_chunks),
    ],
    "Overlap": ["None", "200 chars", "Natural"],
    "Best For": [
        "Uniform documents",
        "Mixed-content corpora (default)",
        "Narrative text (higher cost)",
    ],
})

print("=" * 80)
print("CHUNKING STRATEGY COMPARISON")
print("=" * 80)
print(comparison.to_string(index=False))
print()
log.success("Chunking deep dive complete — recursive chunking with 1000/200 is the recommended default.")

CHUNKING STRATEGY COMPARISON
               Strategy  Chunks  Avg Size (chars)  Min Size  Max Size   Overlap                        Best For
             Fixed-size      15               481       236       500      None               Uniform documents
Recursive (recommended)      10               744       520       992 200 chars Mixed-content corpora (default)
   Semantic (simulated)       8               902       588      1048   Natural    Narrative text (higher cost)

[SUCCESS] Chunking deep dive complete — recursive chunking with 1000/200 is the recommended default.


---

## Summary

- Built a complete RAG pipeline (§6.1): document loading, recursive chunking, embeddings, a FAISS vector store, and grounded question answering with provenance tracking.
- Ran a standard query on RAG limitations and a diagnostic refund-policy query, each returning an answer with cited source passages (pp. 150–152).
- Compared fixed-size, recursive, and semantic chunking side by side; recursive chunking (chunk_size=1000, overlap=200) is the recommended default for mixed-content corpora (p. 151).
- Simulation Mode used chapter-derived mocks (`MockEmbeddings`, `MockRetrievalQAResult`); add `OPENAI_API_KEY` to `.env` to rerun in Live Mode.

**Next in the book:** Chapter 6, §6.2 — Document Intelligence Agents (pp. 153–160), where OCR and schema-driven extraction cross the document boundary.